# 08 - Rule Induction

This notebook evaluates Rule Induction as a separate non-probabilistic
classification family, using RIPPER as its representative algorithm.

The four experimental conditions used in the previous notebooks are retained:
all valid predictors, univariate filtering, multivariate filtering, and
wrapper-based feature selection.

The same dataset, outer cross-validation folds, and evaluation metrics are
maintained so that the results can be included in the final PA1 comparison.

In [1]:
import pandas as pd
import numpy as np
import json
import csv
import io

from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

from wittgenstein import RIPPER

pd.set_option("display.max_columns", None)

## 1. Dataset Preparation

The same dataset preparation used in the previous experiments is maintained.

Identifier and constant attributes are removed, while `Attrition` is encoded
as a binary target:

- `No` -> 0
- `Yes` -> 1

The resulting dataset contains 30 valid original predictors.

In [2]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Preprocessing Definition

RIPPER accepts numerical values and nominal categories directly.

Numerical predictors remain in their original units, and nominal categories
are retained as strings without introducing an artificial ordering.

The implementation learns up to ten bins for numerical predictors with more
than ten distinct values. These thresholds are fitted during training and
reused when predicting the validation fold.

In [3]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical predictors:", len(numerical_features))
print("Categorical predictors:", len(categorical_features))

Numerical predictors: 23
Categorical predictors: 7


In [4]:
def prepare_predictors(X):
    """
    Preserve numerical values and convert nominal columns to object strings.
    """

    X_prepared = X.copy()

    for feature in X_prepared.select_dtypes(exclude=np.number):
        X_prepared[feature] = (
            X_prepared[feature]
            .astype(str)
            .astype(object)
        )

    return X_prepared

## 3. Rule Induction Classifier

RIPPER grows rules by adding conditions that cover positive training cases.
It then prunes conditions using a separate portion of its training data and
optimizes the resulting rule set.

The model predicts attrition when at least one IF-THEN rule matches. Otherwise,
it predicts the negative class. Explicit conditions make the model interpretable.

The `wittgenstein` implementation uses random seed 42 and its remaining default
parameters. Explicit fitting and prediction preserve DataFrame column names
and avoid relying on its compatibility with recent scikit-learn pipelines.

In [5]:
def create_classifier():

    return RIPPER(
        random_state=42
    )


classifier = create_classifier()

classifier

<RIPPER(n_discretize_bins=10, max_rule_conds=None, min_ruleset_samples=None, max_total_conds=None, max_rules=None, k=2, random_state=42, verbosity=0, alpha=1.0, dl_allowance=64, prune_size=0.33)>

## 4. Evaluation Strategy

The same stratified 10-fold cross-validation procedure used in the previous
experiments is maintained.

For each outer fold, preprocessing, feature selection, and classification are
fitted exclusively using the training portion. The validation fold is used
only to estimate predictive performance.

The wrapper performs its internal selection using only the outer training
fold. Numeric discretization is refitted within every internal training split.

In [6]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## 5. Evaluation Metrics

The same evaluation metrics used in the previous experiments are retained:

- Accuracy
- Precision
- Recall
- F1-score
- Balanced Accuracy

The positive class corresponds to `Attrition = Yes`. Mean values and standard
deviations are calculated across the ten outer validation folds.

In [7]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

## 6. Classification Function

A common classification function fits RIPPER on the current training data
and applies its learned rules and discretization to the validation data.

Nominal columns are converted to object strings for compatibility with the
library. Neither scaling nor one-hot encoding is applied by this function.

In [8]:
def fit_predict(X_train, y_train, X_valid):
    """
    Fit RIPPER using training data and predict the validation observations.
    """

    classifier = create_classifier()

    classifier.fit(
        prepare_predictors(X_train).reset_index(drop=True),
        y=y_train.to_numpy(),
        pos_class=1
    )

    predictions = classifier.predict(
        prepare_predictors(X_valid).reset_index(drop=True)
    )

    return np.asarray(predictions, dtype=int), classifier

## 7. Univariate Filter Feature Selection

The univariate experiment retains the existing `SelectKBest` procedure with
ANOVA F-scores and 15 selected transformed features.

Scaling and one-hot encoding are fitted only on the outer training fold for
selection. Selected numerical columns are then restored to their original
units, while selected dummy indicators remain binary.

This adaptation preserves the exact selected components without restoring
unselected categories. A selected component is not always an original variable.

In [9]:
N_SELECTED_FEATURES = 15

print(
    f"Number of transformed features to retain: "
    f"{N_SELECTED_FEATURES}"
)

Number of transformed features to retain: 15


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 8. Multivariate Filter Feature Selection

The same relevance-redundancy selector used in the multivariate experiment
is maintained.

Feature relevance is estimated using mutual information, while redundancy is
measured using absolute correlation with already selected features.

The following implementation is unchanged from notebook 06. It retains 15
transformed features using only the current training fold.

In [11]:
class MRMRSelector(BaseEstimator, TransformerMixin):
    """
    Greedy multivariate filter inspired by mRMR.

    Features are selected by maximizing target relevance while
    penalizing redundancy with features already selected.
    """

    def __init__(
        self,
        k=15,
        random_state=42
    ):
        self.k = k
        self.random_state = random_state

    def fit(self, X, y):

        # Convert sparse matrices generated by one-hot encoding to dense arrays.
        if sparse.issparse(X):
            X_array = X.toarray()
        else:
            X_array = np.asarray(X)

        n_features = X_array.shape[1]

        if self.k > n_features:
            raise ValueError(
                "k cannot be greater than the number of features."
            )

        # Measure the relevance of every feature with respect to the target.
        relevance = mutual_info_classif(
            X_array,
            y,
            random_state=self.random_state
        )

        # Correlation matrix used to quantify redundancy.
        correlation = np.corrcoef(
            X_array,
            rowvar=False
        )

        correlation = np.nan_to_num(
            np.abs(correlation),
            nan=0.0
        )

        selected = []

        # Start with the most relevant feature.
        first_feature = int(np.argmax(relevance))
        selected.append(first_feature)

        # Greedily add features with high relevance and low redundancy.
        while len(selected) < self.k:

            best_feature = None
            best_score = -np.inf

            for feature in range(n_features):

                if feature in selected:
                    continue

                redundancy = np.mean(
                    correlation[feature, selected]
                )

                score = (
                    relevance[feature]
                    - redundancy
                )

                if score > best_score:
                    best_score = score
                    best_feature = feature

            selected.append(best_feature)

        self.selected_indices_ = np.array(selected)
        self.relevance_scores_ = relevance

        return self

    def transform(self, X):

        return X[:, self.selected_indices_]

    def get_support(self):

        return self.selected_indices_

In [12]:
def restore_numerical_features(X_transformed, X_original, feature_names):
    """
    Restore numerical units while retaining selected dummy indicators.
    """

    X_selected = pd.DataFrame(
        X_transformed,
        columns=feature_names,
        index=X_original.index
    )

    for feature in feature_names:

        if feature.startswith("num__"):
            original_feature = feature.split("__", 1)[1]
            X_selected[feature] = X_original[original_feature].to_numpy()

    return X_selected

In [13]:
def select_filter_features(X_train, y_train, X_valid, experiment):
    """
    Fit the existing filter procedure using only the outer training data.
    """

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_valid_transformed = preprocessor.transform(X_valid)

    if sparse.issparse(X_train_transformed):
        X_train_transformed = X_train_transformed.toarray()
        X_valid_transformed = X_valid_transformed.toarray()

    if experiment == "Univariate":
        selector = SelectKBest(
            score_func=f_classif,
            k=N_SELECTED_FEATURES
        )
    else:
        selector = MRMRSelector(
            k=N_SELECTED_FEATURES,
            random_state=42
        )

    selector.fit(X_train_transformed, y_train)

    if experiment == "Univariate":
        selected_indices = np.flatnonzero(selector.get_support())
    else:
        selected_indices = selector.get_support()

    selected_feature_names = (
        preprocessor.get_feature_names_out()[selected_indices].tolist()
    )

    X_train_selected = restore_numerical_features(
        X_train_transformed[:, selected_indices],
        X_train,
        selected_feature_names
    )

    X_valid_selected = restore_numerical_features(
        X_valid_transformed[:, selected_indices],
        X_valid,
        selected_feature_names
    )

    return X_train_selected, X_valid_selected, selected_feature_names

## 9. Wrapper Feature Selection

Sequential Forward Selection evaluates candidate original predictors using
RIPPER, F1 scoring, and internal stratified two-fold cross-validation.

The search is limited to three additions to control computational cost. The
best prefix according to internal mean F1 is retained; ties favor fewer
predictors. This fixed budget has less search breadth and noisier internal
estimates than the existing 15-feature, five-fold Decision Tree wrapper.

Every candidate fits fresh rules and discretization using its internal training
split. The outer validation fold is excluded from the entire search.

In [14]:
N_WRAPPER_FEATURES = 3

inner_cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

In [15]:
def evaluate_subset(X, y, selected_features, inner_splits):
    """
    Estimate the internal F1 of a candidate subset using fresh RIPPER fits.
    """

    scores = []

    for train_indices, valid_indices in inner_splits:

        predictions, classifier = fit_predict(
            X.iloc[train_indices][selected_features],
            y.iloc[train_indices],
            X.iloc[valid_indices][selected_features]
        )

        scores.append(
            f1_score(
                y.iloc[valid_indices],
                predictions,
                zero_division=0
            )
        )

    return float(np.mean(scores))

In [16]:
def select_wrapper_features(X, y):
    """
    Select original predictors using RIPPER and internal cross-validation.
    """

    inner_splits = list(inner_cv.split(X, y))
    selected_features = []
    best_subset = []
    best_score = -np.inf
    selection_trace = []

    for step in range(1, N_WRAPPER_FEATURES + 1):

        candidates = []

        for feature in X.columns:

            if feature in selected_features:
                continue

            candidate_features = selected_features + [feature]

            score = evaluate_subset(
                X,
                y,
                candidate_features,
                inner_splits
            )

            candidates.append((score, feature))

        score, feature = max(candidates, key=lambda item: item[0])
        selected_features.append(feature)

        selection_trace.append({
            "step": step,
            "features": selected_features.copy(),
            "inner_f1": score
        })

        if score > best_score:
            best_score = score
            best_subset = selected_features.copy()

    return best_subset, selection_trace

## 10. Evaluation Function

A common evaluation function applies each experimental condition within the
same outer cross-validation folds.

All features, filter selection, and classifier-specific wrapper selection use
the representations defined above. Fold-level metrics, selected inputs,
learned bins, and rules are retained for inspection.

In [17]:
def calculate_metrics(y_valid, predictions):
    """
    Calculate the same held-out metrics used in the previous experiments.
    """

    return {
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(y_valid, predictions, zero_division=0),
        "Recall": recall_score(y_valid, predictions, zero_division=0),
        "F1": f1_score(y_valid, predictions, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_valid, predictions)
    }

In [18]:
def evaluate_model(experiment, X, y, cv):
    """
    Evaluate RIPPER under one feature strategy using stratified outer folds.
    """

    results = []

    for fold, (train_indices, valid_indices) in enumerate(cv.split(X, y), 1):

        X_train = X.iloc[train_indices]
        y_train = y.iloc[train_indices]
        X_valid = X.iloc[valid_indices]
        selection_trace = []

        if experiment == "All Features":
            X_train_selected = X_train
            X_valid_selected = X_valid
            selected_features = X_train.columns.tolist()

        elif experiment in ["Univariate", "Multivariate"]:
            X_train_selected, X_valid_selected, selected_features = (
                select_filter_features(X_train, y_train, X_valid, experiment)
            )

        else:
            selected_features, selection_trace = select_wrapper_features(
                X_train,
                y_train
            )
            X_train_selected = X_train[selected_features]
            X_valid_selected = X_valid[selected_features]

        predictions, classifier = fit_predict(
            X_train_selected,
            y_train,
            X_valid_selected
        )

        result = {
            "Experiment": experiment,
            "Model": "RIPPER",
            "Fold": fold,
            "Number of Features": len(selected_features),
            "Selected Features": json.dumps(selected_features),
            "Wrapper Trace": json.dumps(selection_trace),
            "Rules": str(classifier.ruleset_),
            "Discretization": json.dumps(
                classifier.bin_transformer_.bins_,
                default=str
            )
        }

        result.update(calculate_metrics(y.iloc[valid_indices], predictions))
        results.append(result)

    return results

## 11. All-Features Model Evaluation

RIPPER is evaluated using all 30 valid original predictors.

Each training fold independently determines the rules and numerical bins.
The same ten outer validation folds used in the previous experiments are
maintained.

In [19]:
print("Evaluating RIPPER using all features...")

baseline_results = evaluate_model(
    experiment="All Features",
    X=X,
    y=y,
    cv=cv
)

print("Evaluation completed.")

Evaluating RIPPER using all features...


Evaluation completed.


## 12. Univariate Model Evaluation

RIPPER is evaluated after applying the existing univariate filter.

The 15 transformed components are selected separately inside each outer
training fold. The full-data descriptive subset from notebook 05 is not used
to estimate performance.

In [20]:
print("Evaluating RIPPER using univariate selection...")

univariate_results = evaluate_model(
    experiment="Univariate",
    X=X,
    y=y,
    cv=cv
)

print("Evaluation completed.")

Evaluating RIPPER using univariate selection...
Evaluation completed.


## 13. Multivariate Model Evaluation

RIPPER is evaluated after applying the existing multivariate filter.

Mutual information, redundancy, and the selected subset are calculated using
only the current training fold. The method is not replaced by CFS.

In [21]:
print("Evaluating RIPPER using multivariate selection...")

multivariate_results = evaluate_model(
    experiment="Multivariate",
    X=X,
    y=y,
    cv=cv
)

print("Evaluation completed.")

Evaluating RIPPER using multivariate selection...
Evaluation completed.


## 14. Wrapper Model Evaluation

RIPPER is evaluated using its own forward-selected original predictors.

The internal search is repeated independently within each outer training fold.
Because candidate subsets require additional classifier fits, this experiment
may take longer than the filter experiments.

In [22]:
print("Evaluating RIPPER using wrapper selection...")

wrapper_results = evaluate_model(
    experiment="Wrapper",
    X=X,
    y=y,
    cv=cv
)

print("Evaluation completed.")

Evaluating RIPPER using wrapper selection...
Evaluation completed.


## 15. Complete Cross-Validation Results

The following tables report the fold-level results and the mean and standard
deviation of each metric across the ten outer validation folds.

Feature counts refer to original predictors for all features and the wrapper,
and transformed components for the filters. Wrapper counts are fold means.

In [23]:
fold_results = pd.DataFrame(
    baseline_results
    + univariate_results
    + multivariate_results
    + wrapper_results
)

display(
    fold_results[
        [
            "Experiment", "Model", "Fold", "Number of Features",
            "Accuracy", "Precision", "Recall", "F1", "Balanced Accuracy"
        ]
    ].round(4)
)

,Experiment,Model,Fold,Number of Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,RIPPER,1,30,0.8095,0.3333,0.1667,0.2222,0.5508
1,All Features,RIPPER,2,30,0.8571,0.6154,0.3333,0.4324,0.6463
2,All Features,RIPPER,3,30,0.8231,0.3333,0.0833,0.1333,0.5254
3,All Features,RIPPER,4,30,0.8163,0.4000,0.2500,0.3077,0.5884
4,All Features,RIPPER,5,30,0.8639,0.6667,0.3333,0.4444,0.6504
5,All Features,RIPPER,6,30,0.8027,0.3077,0.1667,0.2162,0.5467
6,All Features,RIPPER,7,30,0.8571,0.6364,0.2917,0.4000,0.6296
7,All Features,RIPPER,8,30,0.8231,0.4000,0.2609,0.3158,0.5941
8,All Features,RIPPER,9,30,0.8639,0.7143,0.2174,0.3333,0.6006
9,All Features,RIPPER,10,30,0.7891,0.1000,0.0435,0.0606,0.4854


In [24]:
metric_names = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "balanced_accuracy": "Balanced Accuracy"
}

results = []

for experiment in ["All Features", "Univariate", "Multivariate", "Wrapper"]:

    experiment_results = fold_results[
        fold_results["Experiment"] == experiment
    ]

    result = {
        "Experiment": experiment,
        "Model": "RIPPER",
        "Selected Features": experiment_results["Number of Features"].mean()
    }

    for metric in scoring.keys():

        metric_values = experiment_results[metric_names[metric]]
        result[f"{metric}_mean"] = metric_values.mean()
        result[f"{metric}_std"] = metric_values.std(ddof=0)

    results.append(result)

ripper_results = pd.DataFrame(results)

display(
    ripper_results.round(4)
)

,Experiment,Model,Selected Features,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,All Features,RIPPER,30.0,0.8306,0.0263,0.4507,0.1881,0.2147,0.0942,0.2866,0.1209,0.5818,0.0513
1,Univariate,RIPPER,15.0,0.8415,0.0193,0.6102,0.2783,0.1558,0.0948,0.2316,0.1059,0.5645,0.0454
2,Multivariate,RIPPER,15.0,0.8429,0.0198,0.5439,0.2656,0.1558,0.1170,0.2280,0.1413,0.5653,0.0554
3,Wrapper,RIPPER,2.6,0.7823,0.0386,0.3640,0.0934,0.4687,0.1735,0.3996,0.1132,0.6557,0.0754


## 16. Rule Induction Performance Summary

The mean predictive performance of RIPPER is summarized across the four
feature strategies.

The metric names and their order are unchanged from the previous experiments,
allowing these rows to be included in the existing comparison files.

In [25]:
summary_results = ripper_results[
    [
        "Experiment",
        "Model",
        "Selected Features",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "balanced_accuracy_mean"
    ]
].copy()

summary_results.columns = [
    "Experiment",
    "Model",
    "Selected Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy"
]

summary_results = summary_results.sort_values(
    "F1",
    ascending=False
)

display(
    summary_results.round(4)
)

,Experiment,Model,Selected Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
3,Wrapper,RIPPER,2.6,0.7823,0.3640,0.4687,0.3996,0.6557
0,All Features,RIPPER,30.0,0.8306,0.4507,0.2147,0.2866,0.5818
1,Univariate,RIPPER,15.0,0.8415,0.6102,0.1558,0.2316,0.5645
2,Multivariate,RIPPER,15.0,0.8429,0.5439,0.1558,0.2280,0.5653


## 17. Export Results

The RIPPER rows are added to the existing result and summary files for each
strategy. Previously recorded rows for other classifiers are preserved.

Fold-level inputs, rules, bins, and wrapper selections are exported separately
for inspection. Repeating this notebook replaces only its own RIPPER rows.
If notebooks 04-07 are rerun, execute this notebook again before final analysis.

In [26]:
def update_results(file_path, result):
    """
    Replace only RIPPER rows while preserving the original CSV rows.
    """

    with open(file_path, "r", encoding="utf-8", newline="") as file:
        lines = file.readlines()

    columns = next(csv.reader([lines[0]]))
    model_index = columns.index("Model")
    retained_lines = [lines[0]]

    for line in lines[1:]:
        if next(csv.reader([line]))[model_index] != "RIPPER":
            retained_lines.append(line)

    output = io.StringIO(newline="")
    writer = csv.DictWriter(
        output,
        fieldnames=columns,
        lineterminator="\n"
    )
    writer.writerow({column: result.get(column, "") for column in columns})

    original_rows = "".join(retained_lines)

    if not original_rows.endswith("\n"):
        original_rows += "\n"

    with open(file_path, "w", encoding="utf-8", newline="") as file:
        file.write(original_rows + output.getvalue())

In [27]:
fold_results.to_csv(
    "../../results/pa1/ripper_fold_results.csv",
    index=False
)

experiment_files = {
    "All Features": "baseline",
    "Univariate": "univariate",
    "Multivariate": "multivariate",
    "Wrapper": "wrapper"
}

for experiment, file_name in experiment_files.items():

    result = ripper_results.loc[
        ripper_results["Experiment"] == experiment
    ].iloc[0].to_dict()

    summary = summary_results.loc[
        summary_results["Experiment"] == experiment
    ].iloc[0].to_dict()

    update_results(
        f"../../results/pa1/{file_name}_results.csv",
        result
    )

    update_results(
        f"../../results/pa1/{file_name}_summary.csv",
        summary
    )

print("Rule induction results saved successfully.")

Rule induction results saved successfully.


## 18. Comparison with Previous Experiments

The RIPPER results are compared with the existing classifiers under the four
feature strategies.

All configurations use the same outer folds and metrics. The classifier input
representations and wrapper budgets differ as described above.

In [28]:
comparison_results = []

for experiment, file_name in experiment_files.items():

    experiment_summary = pd.read_csv(
        f"../../results/pa1/{file_name}_summary.csv"
    )
    experiment_summary["Experiment"] = experiment
    comparison_results.append(experiment_summary)

comparison = pd.concat(
    comparison_results,
    ignore_index=True
)

comparison = comparison[
    [
        "Experiment",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy"
    ]
]

display(
    comparison.round(4)
)

,Experiment,Model,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,0.8490,0.5492,0.4219,0.4738,0.6765
1,All Features,Decision Tree,0.7925,0.3604,0.3670,0.3620,0.6206
2,All Features,SVM,0.8653,0.8363,0.2241,0.3461,0.6064
3,All Features,K-NN,0.8442,0.5412,0.1350,0.2115,0.5577
4,All Features,RIPPER,0.8306,0.4507,0.2147,0.2866,0.5818
5,Univariate,Decision Tree,0.7789,0.3546,0.4219,0.3809,0.6347
6,Univariate,Neural Network,0.8163,0.4167,0.3293,0.3642,0.6197
7,Univariate,K-NN,0.8463,0.5761,0.2062,0.2968,0.5877
8,Univariate,SVM,0.8571,0.7933,0.1692,0.2737,0.5793
9,Univariate,RIPPER,0.8415,0.6102,0.1558,0.2316,0.5645


## 19. Main Observations

The four experiments show that the effect of feature selection also depends
on the classifier within the Rule Induction family.

- **RIPPER with all predictors** obtains F1 **0.2866** and balanced accuracy
  **0.5818**, using all 30 valid original predictors.

- The **univariate filter** obtains F1 **0.2316** and balanced accuracy
  **0.5645**. The **multivariate filter** obtains F1 **0.2280** and balanced
  accuracy **0.5653**. Both filters retain 15 transformed components and reduce
  RIPPER's F1 relative to its complete representation.

- The **RIPPER wrapper** obtains F1 **0.3996** and balanced accuracy **0.6557**.
  It retains a mean of **2.6 original predictors**, with two or three selected
  in each outer fold. This is the strongest reduced-feature PA1 configuration.

- The **Neural Network using all features** remains the strongest overall PA1
  configuration, with F1 **0.4738** and balanced accuracy **0.6765**. Adding
  Rule Induction does not change the best observed PA1 F1-score.

- Original predictors and selected transformed components are different units.
  The bounded wrapper also uses fewer internal folds and search steps than
  the existing Decision Tree wrapper, limiting direct comparisons of selection.

The final analysis combines these results with the other classifier families.
The reported means describe the evaluated configurations and do not establish
statistical significance.